In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import train_test_split
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor


pd.set_option('display.max_columns', None)
df = pd.read_csv('Housing Data.csv')

#----------DESCRIPTIVE STATISTICS OUTPUT ------------------------------------------------------------------------


#descriptive stats output along with range & variance outputs
columns = ['Price', 'SquareFootage', 'NumBathrooms', 'NumBedrooms', 'AgeOfHome', 'Floors', 'SchoolRating', 'CrimeRate', 'DistanceToCityCenter', 'LocalAmenities']

stats = df[columns].describe()
stats.loc['range'] = df[columns].max() - df[columns].min()
stats.loc['variance'] = df[columns].var()

print(stats)
print()


#Garage categorical outputs as frequency counts and percentage 
print(df['Garage'].value_counts())
print()
print(df['Garage'].value_counts(normalize=True) * 100)


#---------UNIVARIATE VISUALIZATIONS --------------------------------------------------------------------------------------------

quantitative_columns = ['Price', 'SquareFootage', 'NumBathrooms', 'NumBedrooms', 'AgeOfHome', 
              'Floors', 'SchoolRating', 'CrimeRate', 'DistanceToCityCenter', 'LocalAmenities']

fig, axes = plt.subplots(4, 3, figsize=(15, 16))
axes = axes.flatten()

for i, col in enumerate(quantitative_columns):
    axes[i].hist(df[col], bins=30, edgecolor='black')
    axes[i].set_title(col)
    axes[i].set_xlabel(col)
    axes[i].set_ylabel('Frequency')

#garage bar chart addition
df['Garage'].value_counts().plot(kind='bar', ax=axes[len(quantitative_columns)])
axes[len(quantitative_columns)].set_title('Garage')
axes[len(quantitative_columns)].set_xlabel('Garage')
axes[len(quantitative_columns)].set_ylabel('Count')


#hides 12th empty grid
for j in range(len(quantitative_columns)+1, len(axes)):
    fig.delaxes(axes[j])

plt.tight_layout()
plt.subplots_adjust(hspace=0.6)
plt.savefig('univariate_visualizations.png', dpi=500)



#-----BIVARIATE VISUALIZATIONS WITH PRICE -----------------------


selected_iv = ['SquareFootage', 'NumBathrooms', 'NumBedrooms', 'AgeOfHome', 
           'Floors', 'SchoolRating', 'CrimeRate', 'DistanceToCityCenter', 'LocalAmenities']

discrete_columns = ['Floors', 'SchoolRating', 'LocalAmenities', 'NumBathrooms', 'NumBedrooms']

fig, axes = plt.subplots(4, 3, figsize=(15, 16))
axes = axes.flatten()

for i, col in enumerate(selected_iv):
    if col in discrete_columns:
        jitter = np.random.normal(0, 0.1, size=len(df))
        axes[i].scatter(df[col] + jitter, df['Price'], alpha=0.15, s=10)
    else:
        axes[i].scatter(df[col], df['Price'], alpha=0.3)
    axes[i].set_title(f'{col}')
    axes[i].set_xlabel(col)
    axes[i].set_ylabel('Price')

#Garage boxplot added
df.boxplot(column='Price', by='Garage', ax=axes[len(selected_iv)])
axes[len(selected_iv)].set_title('Price by Garage')

#hides 12th empty grid
for j in range(len(selected_iv)+1, len(axes)):
    fig.delaxes(axes[j])

plt.suptitle('')
plt.tight_layout()
plt.subplots_adjust(hspace=0.6)
plt.savefig('bivariate_visualizations.png', dpi=500)

#-------DATA CLEANING-----------------------------------------------
print('CLEANING... DETECTION')

#check for duplicate rows in the dataset
duplicate_count = df.duplicated().sum()
print("\ntotal duplicate rows found:", duplicate_count)

#check each column for missing entries
print("\nnull value count by column:")
print(df.isnull().sum())


#check selected columns for negative/illogical values
columns = ['Price', 'SquareFootage', 'NumBathrooms', 'NumBedrooms', 'AgeOfHome', 
           'Floors', 'SchoolRating', 'CrimeRate', 'DistanceToCityCenter', 'LocalAmenities']

print("\nnegative value count:")
for col in columns:
    print(f"{col}: {(df[col] < 0).sum()} negative values")


#check categorical column(s) for inconsistent label formatting
categorical_columns = ['Garage', 'HouseColor', 'Fireplace', 'IsLuxury']

print("\nUnique category values by column:")
for col in categorical_columns:
    print(f" {col} ")
    print(df[col].unique())


#check column names for formatting issues
print("\nColumn names:", list(df.columns))

#check data types for each column
print("\nData type variations:")
print(df.dtypes)


#outlier detection using IQR method
numeric_columns = ['SquareFootage', 'NumBathrooms', 'NumBedrooms', 'AgeOfHome',
                'Floors', 'SchoolRating', 'CrimeRate', 'DistanceToCityCenter',
                'LocalAmenities']

for col in  numeric_columns:    
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower_boundary = Q1 - 1.5 * IQR
    upper_boundary = Q3 + 1.5 * IQR
    outliers = df[(df[col] < lower_boundary) | (df[col] > upper_boundary)]
    print(f"{col}: {len(outliers)} outliers (bounds: {lower_boundary:.2f} to {upper_boundary:.2f})")

  #encoding garage as binary
df['Garage'] = df['Garage'].map({'Yes': 1, 'No': 0})

print("\nGarage results:")
print(df['Garage'].value_counts())
print(df['Garage'].dtype)

#mitigate outliers by capping IQR 

#note: SchoolRating and Floors are not capped as they are within valid ranges. 
cols_to_cap = ['SquareFootage', 'AgeOfHome', 'CrimeRate', 'DistanceToCityCenter']

for col in cols_to_cap:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower_bounds = Q1 - 1.5 * IQR
    upper_bounds = Q3 + 1.5 * IQR
    df[col] = df[col].clip(lower=lower_bounds, upper=upper_bounds)

    #export of cleaned dataset
    
cleaned_columns = ['Price', 'SquareFootage', 'NumBathrooms', 'NumBedrooms', 'Garage', 
                    'AgeOfHome', 'Floors', 'SchoolRating', 'CrimeRate', 
                    'DistanceToCityCenter', 'LocalAmenities']

df[cleaned_columns].to_csv('cleaned_dataset.csv', index=False)

#---------------------SPLIT-----------------

#define independent variables and dependent variable
independent = df[['SquareFootage', 'NumBathrooms', 'NumBedrooms', 'Garage', 'AgeOfHome', 
                   'Floors', 'SchoolRating', 'CrimeRate', 'DistanceToCityCenter', 'LocalAmenities']]
dependent = df['Price']

#70/30 train/test split
independent_train, independent_test, dependent_train, dependent_test = train_test_split(
    independent, dependent, test_size=0.3, random_state=100
)

print("Training shape:", independent_train.shape)
print("Testing shape:", independent_test.shape)


#exporting train/test datasets 
train_export = independent_train.copy()
train_export['Price'] = dependent_train
train_export.to_csv('training_dataset.csv', index=False)

test_export = independent_test.copy()
test_export['Price'] = dependent_test
test_export.to_csv('testing_dataset.csv', index=False)

#-----------------------STATISTICAL METHOD ----------------------------

#adds intercept to the independent variables
independent_train_constant = sm.add_constant(independent_train)

#builds multiple linear regression model on the training data
model = sm.OLS(dependent_train, independent_train_constant).fit()

#model summary
print(model.summary())



#--------------------BACKWARD STEPWISE ELIMINATION OPTIMIZATION----------------------------------

#step 1: remove CrimeRate with highest p-value 
independent_train_opt = independent_train.drop(columns=['CrimeRate'])
independent_train_opt_constant = sm.add_constant(independent_train_opt)
model_opt = sm.OLS(dependent_train, independent_train_opt_constant).fit()


#step 2: remove Garage 
independent_train_opt = independent_train_opt.drop(columns=['Garage'])
independent_train_opt_constant = sm.add_constant(independent_train_opt)
model_opt = sm.OLS(dependent_train, independent_train_opt_constant).fit()


#step 3: remove Floors 
independent_train_opt = independent_train_opt.drop(columns=['Floors'])
independent_train_opt_constant = sm.add_constant(independent_train_opt)
model_opt = sm.OLS(dependent_train, independent_train_opt_constant).fit()


#step 4: remove SchoolRating 
independent_train_opt = independent_train_opt.drop(columns=['SchoolRating'])
independent_train_opt_constant = sm.add_constant(independent_train_opt)
model_opt = sm.OLS(dependent_train, independent_train_opt_constant).fit()

print("\nOptimized model summary:\n")
print(model_opt.summary())

print("\nBREAKPOINT - OPTIMIZATION COMPLETE\n")

#--------PREDICTION ON TEST SET ----------------------

#6 variables selected
final_variables = ['SquareFootage', 'NumBathrooms', 'NumBedrooms', 'AgeOfHome', 'DistanceToCityCenter', 'LocalAmenities']

independent_test_final = independent_test[final_variables]
independent_test_final_constant = sm.add_constant(independent_test_final)

#predicts Price
predictions = model_opt.predict(independent_test_final_constant)

#calculates RMSE
rmse = np.sqrt(mean_squared_error(dependent_test, predictions))
print("RMSE:", rmse)

print("\nParameters for equation:\n")
print(model_opt.params)


#------------ASSUMPTIONS ---------------------------------------------


#linearity check: residuals vs fitted values
residuals = model_opt.resid
fitted_values = model_opt.fittedvalues

plt.figure(figsize=(8,6))
plt.scatter(fitted_values, residuals, alpha=0.3)
plt.axhline(y=0, color='red', linestyle='--')
plt.title('Residuals vs Fitted Values')
plt.xlabel('Fitted Values')
plt.ylabel('Residuals')
plt.savefig('linearityD5.png', dpi=300)


#variance inflation factor on the 6 model variables
vif_data = pd.DataFrame()
vif_data['Variable'] = independent_train_opt.columns
vif_data['VIF'] = [variance_inflation_factor(independent_train_opt.values, i) for i in range(independent_train_opt.shape[1])]

print(vif_data)